# InferStack N0/N1: native vs vLLM vs SGLang on one A100 or L4

Runtime > Change runtime type > **A100 GPU** (preferred) or **L4 GPU**, then run the cells in order. T4 is refused: it has no BF16, so it would be a different experiment. Results are labelled with the GPU that actually ran (`a100` or `l4`); L4 numbers are slower in absolute terms and are never compared directly against A100 numbers. Uses the pinned `Qwen/Qwen3.5-2B-Base@b1485b2` (BF16) for every engine. No paid optimizer API is used.

What it does, **one engine at a time in this one session**: set up the native environment, then for each engine (native unfused, native full, vLLM, SGLang) install it, qualify it and run a ~2-minute smoke test, printing PASS or FAIL before moving on. A failed stage only needs that cell re-run. Engines never share the GPU: each server is stopped and GPU memory is checked before the next starts. Finally the fair measurement runs every engine that passed the same workload matrix (interactive, moderate, prefill-heavy, shared-prefix × concurrency 1/4/8/16) against each engine for 3 rounds in rotated order. Every raw request is kept.

Expect roughly 2–4 hours (mostly installs and model loads). The result is a **measurement**: relations need 3 rounds with non-overlapping ranges, and quality is evaluated separately. Prepared and CPU-tested with fake engines; not GPU-executed by the author. Send back the ZIP from the last cell.

## 1. Verify GPU, clone privately, pin source

In [ ]:
import base64, datetime, getpass, json, os, signal, subprocess, sys, time
from pathlib import Path
PLATFORM = 'colab'
COMMIT = '787c5678d4d0a2a92513852829877c24b62d1e3b'
BASE = Path('/content' if PLATFORM == 'colab' else '/kaggle/working')
REPO = BASE / 'inferstack'
assert (3, 11) <= sys.version_info[:2] < (3, 14), 'This profile needs Python 3.11–3.13.'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
print(gpu)
if PLATFORM == 'colab':
    # One GPU, BF16-capable (compute capability >= 8.0). A100 preferred; L4 accepted.
    # The label goes into every output path so an L4 run can never be filed as A100.
    GPU_LABEL = 'a100' if 'A100' in gpu else 'l4' if 'L4' in gpu else None
    assert GPU_LABEL, 'Choose an A100 or L4 runtime. T4 has no BF16; use the separate T4 notebook for Kaggle.'
else:
    assert len([line for line in gpu.splitlines() if 'T4' in line]) >= 2, 'Choose Kaggle GPU T4 x2.'
    GPU_LABEL = 't4x2'
print('GPU label:', GPU_LABEL)

def clone_pinned():
    if not (REPO / '.git').is_dir():
        if PLATFORM == 'kaggle':
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
        else:
            token = getpass.getpass('Read-only GitHub token for this private repo: ').strip()
        env = os.environ.copy()
        env.update(GIT_TERMINAL_PROMPT='0', GIT_CONFIG_COUNT='1',
                   GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0='AUTHORIZATION: basic '+base64.b64encode(('x-access-token:'+token).encode()).decode())
        result = subprocess.run(['git','clone','-q','https://github.com/sanjoxtech/inferstack.git',str(REPO)],
                                env=env,capture_output=True,text=True)
        del token, env
        if result.returncode:
            raise RuntimeError('Clone failed. Check read-only repository access and Internet connectivity. Credentials were not printed or stored.')
    subprocess.run(['git','checkout','--detach',COMMIT],cwd=REPO,check=True,capture_output=True,text=True)
    actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
    assert actual == COMMIT
    print('Pinned source:', actual)
clone_pinned()
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=REPO/'results'/f'{PLATFORM}-{GPU_LABEL}-scorecard-{COMMIT[:7]}-{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'gpu-inventory.txt').write_text(gpu)
(OUT/'run-context.json').write_text(json.dumps({'commit':COMMIT,'platform':PLATFORM,'gpu_label':GPU_LABEL,'run_id':RUN_ID},indent=2))

def logged(argv, filename, *, env=None, timeout=3600):
    path=OUT/filename
    start=time.monotonic()
    with path.open('w') as log:
        proc=subprocess.Popen(list(map(str,argv)),cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while proc.poll() is None:
                elapsed=time.monotonic()-start
                if elapsed > timeout:
                    raise TimeoutError(f'Wall-time cap reached: {filename}')
                print(f'{filename}: running, {int(elapsed)} seconds elapsed',flush=True)
                time.sleep(20)
        except BaseException:
            try:
                os.killpg(proc.pid,signal.SIGTERM)
                proc.wait(timeout=20)
            except (ProcessLookupError,subprocess.TimeoutExpired):
                try: os.killpg(proc.pid,signal.SIGKILL)
                except ProcessLookupError: pass
            raise
    print(filename,'exit:',proc.returncode)
    if proc.returncode:
        print(path.read_text(errors='replace')[-3000:])
    return proc.returncode


## 2. Native InferStack environment

In [ ]:
# Reuse the notebook's CUDA-enabled torch; keep changes in a separate virtualenv.
# Colab/Kaggle images may lack the venv package for their current Python (e.g. 3.13):
# install the matching pythonX.Y-venv if creating a virtualenv fails.
def ensure_venv():
    probe=subprocess.run([sys.executable,'-m','venv','--help'],capture_output=True)
    ok=subprocess.run([sys.executable,'-c','import ensurepip'],capture_output=True).returncode==0
    if probe.returncode==0 and ok: return
    assert os.geteuid()==0, 'venv/ensurepip missing and no root to install it'
    pkg=f'python{sys.version_info.major}.{sys.version_info.minor}-venv'
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq',pkg],check=True)
ensure_venv()
VENV=Path('/tmp/inferstack-native-env')
subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(VENV)],check=True)
PY=VENV/'bin/python'
ENV={**os.environ,'PATH':str(VENV/'bin')+os.pathsep+os.environ['PATH'],
     'PYTHONPATH':str(REPO/'src'),'INFERSTACK_PROVIDER':'colab','CUDA_VISIBLE_DEVICES':'0'}
assert logged([PY,'-m','pip','install','--require-hashes','-r','requirements-dev.lock'],
              'install-core.log',env=ENV,timeout=2700)==0
assert logged([PY,'-m','pip','install','--no-cache-dir','-e','.[dev,native,hf,postgres]',
               'transformers[kernels]==5.22.2'], 'install-native.log',env=ENV,timeout=2700)==0
preflight="""import json, torch, triton, transformers
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'transformers':transformers.__version__,
                  'triton':triton.__version__,'gpu':torch.cuda.get_device_name(0)},indent=2))
"""
assert logged([PY,'-c',preflight],'preflight-native.log',env=ENV,timeout=180)==0
print((OUT/'preflight-native.log').read_text())
(OUT/'pip-freeze-native.txt').write_text(subprocess.check_output([str(PY),'-m','pip','freeze'],env=ENV,text=True))


## 3. Stage helpers
Defines the per-engine steps. Run it once; it does not start any engine.

In [ ]:
# Stage helpers: each engine is installed, qualified and smoke-tested on its own, one at a time.
import shutil
PROFILES = 'configs/scorecard/a100-qwen3.5-2b.json'   # no A100-specific setting inside (BF16, 8192 context)
MODEL_REVISION = 'b1485b2fa6dfa1287294f269f5fb618e03d52d7c'
VLLM_SPEC = 'vllm'           # set exact versions here for a pinned rerun; installed versions are recorded
SGLANG_SPEC = 'sglang[all]'
ENGINE_VENVS = {'vllm': (Path('/content/venv-vllm'), VLLM_SPEC), 'sglang': (Path('/content/venv-sglang'), SGLANG_SPEC)}
# SGLang pins outlines==0.1.11 -> outlines_core 0.1.x, which has no Python 3.13 wheel (Colab is 3.13).
ENGINE_PYTHON = {'vllm': None, 'sglang': '3.12'}
# Qualify with the same context cap the scorecard profiles use; without it vLLM sizes for
# 262144 tokens and runs out of memory on a 24 GB L4 (seen 2026-11-10).
CONFORMANCE_EXTRA = {'vllm': '--extra=--max-model-len=8192', 'sglang': '--extra=--context-length=8192'}
# Engine venv bins follow the native venv on PATH, so engine helpers such as ninja (FlashInfer JIT
# in SGLang) are found while native commands still resolve first (missing ninja seen 2026-11-10).
SCORE_ENV = {**ENV, 'PATH': os.pathsep.join([str(VENV/'bin')] + [str(v/'bin') for v, _ in ENGINE_VENVS.values()]
                                             + ENV['PATH'].split(os.pathsep)[1:])}
SMOKE_WORKLOAD = OUT/'smoke-workload.json'
SMOKE_WORKLOAD.write_text(json.dumps({
    'version': 1,
    'cases': [{'name': 'interactive', 'prompt_words': 48, 'max_tokens': 32},
              {'name': 'prefill_heavy', 'prompt_words': 1500, 'max_tokens': 16}],
    'concurrency': [1, 4], 'requests_per_cell': {'min': 4, 'per_concurrency': 1},
    'warmup_requests': 1, 'temperature': 0.0, 'seed': 20261109}, indent=2))
STAGE = {}   # profile -> 'passed' | reason it failed
# vLLM 0.33 on L4 took 858 s to become ready (it profiles the unused vision encoder with OOM retries,
# seen 2026-11-11); the scorecard default of 1200 s leaves too little margin.
READY_TIMEOUT = 2400
WAIVERS = {}  # engine -> conformance checks waived, recorded in waivers.json

def gpu_used_mib():
    out = subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'], text=True)
    return int(out.split()[0])

def uv():
    tools = Path('/content/venv-tools')
    if not (tools/'bin/uv').exists():
        subprocess.run([sys.executable,'-m','venv',str(tools)],check=True)
        subprocess.run([tools/'bin/python','-m','pip','install','-q','uv'],check=True)
    return str(tools/'bin/uv')

def venv_python(venv):
    exe = venv/'bin/python'
    if not exe.exists(): return None
    return subprocess.check_output([str(exe),'-c','import sys;print(f"{sys.version_info[0]}.{sys.version_info[1]}")'],text=True).strip()

def install_engine(name):
    venv, spec = ENGINE_VENVS[name]
    want, have = ENGINE_PYTHON[name], venv_python(venv)
    if have and want and have != want:
        shutil.rmtree(venv); have = None      # e.g. a 3.13 venv left by an earlier attempt
    if not have:
        if want: subprocess.run([uv(),'venv','--seed','--python',want,str(venv)],check=True)
        else: subprocess.run([sys.executable,'-m','venv',str(venv)],check=True)
    print(name, 'venv Python', venv_python(venv))
    code = logged([venv/'bin/python','-m','pip','install','--upgrade','pip',spec,'httpx'], f'install-{name}.log', timeout=3600)
    if code: return f'install failed (see install-{name}.log)'
    (OUT/f'pip-freeze-{name}.txt').write_text(subprocess.check_output([str(venv/'bin/python'),'-m','pip','freeze'],text=True))
    return None

def qualify(name):
    venv, _ = ENGINE_VENVS[name]
    env = {**ENV, 'PATH': str(venv/'bin')+os.pathsep+ENV['PATH']}
    code = logged([PY,'-u','scripts/engine_conformance.py','--engine',name,'--model','Qwen/Qwen3.5-2B-Base',
                   '--revision',MODEL_REVISION,CONFORMANCE_EXTRA[name],'--out',
                   str((OUT/f'conformance-{name}.json').relative_to(REPO))], f'conformance-{name}.runner.log', env=env, timeout=3600)
    if code == 0: return None
    report = json.loads((OUT/f'conformance-{name}.json').read_text()) if (OUT/f'conformance-{name}.json').exists() else {}
    failed = [k for k, v in (report.get('checks') or {}).items() if not v]
    # Waiver: only server-side metric families are missing (e.g. vLLM 0.33 dropped the TPOT histogram
    # name at pinned commit 787c567). The scorecard measures latency client-side, so it is recorded, not fatal.
    if failed == ['metric_families_present']:
        WAIVERS[name] = {'missing_metric_families': report.get('metric_families_missing')}
        (OUT/'waivers.json').write_text(json.dumps(WAIVERS, indent=2))
        print(f'{name}: conformance passed except missing metric families {report.get("metric_families_missing")}; waived and recorded')
        return None
    return f'conformance failed {failed} (see conformance-{name}.log)'

def smoke(profile):
    """Launch exactly the scorecard argv/env for one profile, send a few requests, stop it."""
    before = gpu_used_mib()
    run_id = f'smoke-{profile}-{int(time.time())}'
    logged([PY,'-u','scripts/engine_scorecard.py','run','--profiles',PROFILES,'--workload',str(SMOKE_WORKLOAD),
            '--out',str((OUT/'smoke').relative_to(REPO)),'--rounds','1','--run-id',run_id,'--only',profile,'--ready-timeout',str(READY_TIMEOUT)],
           f'{run_id}.runner.log', env=SCORE_ENV, timeout=3600)
    rdir = OUT/'smoke'/run_id/'round-0'/profile
    st = json.loads((rdir/'status.json').read_text()) if (rdir/'status.json').exists() else {}
    rows = [json.loads(l) for l in (rdir/'requests.jsonl').read_text().splitlines()] if (rdir/'requests.jsonl').exists() else []
    ok = sum(1 for r in rows if r.get('success'))
    time.sleep(5)
    after = gpu_used_mib()
    print(f'{profile}: ready={st.get("ready")} ready_s={st.get("ready_s")} requests ok {ok}/{len(rows)} '
          f'GPU used before {before} MiB, after stop {after} MiB')
    if after > before + 1024:
        print(f'WARNING: {after-before} MiB still in use after stopping {profile}; a leftover process may skew the next engine.')
    if not st.get('ready'):
        log = rdir/'server.log'
        print(log.read_text(errors='replace')[-3000:] if log.exists() else (OUT/f'{run_id}.runner.log').read_text()[-3000:])
        return f'not ready: {st.get("error")}'
    if not rows or ok != len(rows):
        bad = next((r.get('error') for r in rows if not r.get('success')), 'no requests recorded')
        return f'{len(rows)-ok} of {len(rows)} requests failed: {bad}'
    return None

def stage(profile, engine=None):
    reason = None
    if engine:
        reason = install_engine(engine) or qualify(engine)
    reason = reason or smoke(profile)
    STAGE[profile] = 'passed' if reason is None else reason
    print(('PASS ' if reason is None else 'FAIL ')+profile+('' if reason is None else ': '+reason))
    print('stages so far:', STAGE)


## 4a. Native unfused: smoke test

In [ ]:
stage('native_unfused')


## 4b. Native full: smoke test

In [ ]:
stage('native_full')


## 4c. vLLM: install, qualify, smoke test
If this FAILs, send the printed tail (or the ZIP); you can still continue with the next stage.

In [ ]:
stage('vllm', engine='vllm')


## 4d. SGLang: install, qualify, smoke test

In [ ]:
stage('sglang', engine='sglang')


## 5. Fair measurement: every engine that passed, rotated, 3 rounds
Same GPU, same pinned weights, same requests, execution order rotated each round. QUICK=True runs one round (no relations claimed).

In [ ]:
QUICK = False
ROUNDS = 1 if QUICK else 3
only = [p for p in ('native_unfused','native_full','vllm','sglang') if STAGE.get(p) == 'passed']
print('profiles to run:', only, '| skipped:', {p: r for p, r in STAGE.items() if r != 'passed'})
assert 'native_unfused' in only, 'The baseline must pass its smoke test first.'
# L4 has roughly a quarter of A100 memory bandwidth, so the wall-time cap is larger there.
code = logged([PY,'-u','scripts/engine_scorecard.py','run','--profiles',PROFILES,
               '--out',str((OUT/'scorecard').relative_to(REPO)),'--rounds',str(ROUNDS),'--run-id',GPU_LABEL,
               '--only',*only,'--ready-timeout',str(READY_TIMEOUT)], 'scorecard-runner.log', env=SCORE_ENV, timeout=(5 if GPU_LABEL == 'a100' else 10)*60*60)
print('scorecard exit:', code)
(OUT/'stages.json').write_text(json.dumps(STAGE, indent=2))
md = OUT/'scorecard'/GPU_LABEL/'scorecard.md'
print(md.read_text() if md.exists() else 'no scorecard: inspect scorecard-runner.log')


## 6. Download evidence

In [ ]:
import shutil
archive_base=BASE/f'inferstack-{PLATFORM}-{GPU_LABEL}-evidence-{RUN_ID}'
zip_path=shutil.make_archive(str(archive_base),'zip',root_dir=OUT.parent,base_dir=OUT.name)
print('Evidence archive:',zip_path)
if PLATFORM=='colab':
    from google.colab import files
    files.download(zip_path)
else:
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
    print('You can also download the ZIP from the notebook file browser under /kaggle/working.')
